# Feature Importance Analysis

This notebook consolidates all features proposed by teammates and evaluates their importance using LightGBM. The goal is to identify which features are genuinely useful, which are redundant, and which should be dropped before the next model run.

**Feature sources:**
- **Tom / Model_Pipeline:** `lag_1/7/28`, `rmean_3/7/28`, `price_change`, `price_mean_7`, calendar features, hierarchical features
- **Feature_Engineering_V1:** `lag_14/35/42/364/365`, `rolling_std_7`, `price_vs_mean`, `price_momentum`, `is_high_impact_event`, `snap`

**Key methodological note:** We use a **time-based train/val split** (not random) to avoid data leakage — since lag and rolling features depend on past values, a random split would allow future information to leak into training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMRegressor
import lightgbm as lgb

## 1. Load Data

In [2]:
import os
DATA_PATH = r'C:\Users\user\Predictive-Analytics-Project\data'  # adjust to your path

sales      = pd.read_csv(os.path.join(DATA_PATH, 'sales_train_validation.csv'))
calendar   = pd.read_csv(os.path.join(DATA_PATH, 'calendar.csv'))
prices     = pd.read_csv(os.path.join(DATA_PATH, 'sell_prices.csv'))

print('sales shape   :', sales.shape)
print('calendar shape:', calendar.shape)
print('prices shape  :', prices.shape)

sales shape   : (30490, 1919)
calendar shape: (1969, 14)
prices shape  : (6841121, 4)


## 2. Melt & Merge

We use only the last 500 days of data (same as the team's current pipeline) to manage memory. This is an important constraint to keep in mind when interpreting results — we discuss its implications in Section 9.

In [3]:
N_DAYS  = 500
day_cols = [f'd_{i}' for i in range(1914 - N_DAYS, 1914)]
id_cols  = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

df = sales[id_cols + day_cols].melt(
    id_vars=id_cols,
    var_name='d',
    value_name='sales'
)

cal_cols = ['d', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
            'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
            'snap_CA', 'snap_TX', 'snap_WI']
df = df.merge(calendar[cal_cols], on='d', how='left')
df['date'] = pd.to_datetime(df['date'])
df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

df['sales']      = df['sales'].astype('float32')
df['sell_price'] = df['sell_price'].astype('float32')

for col in ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1', 'event_type_1']:
    df[col] = df[col].astype('category')

df = df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)
print('Merged df shape:', df.shape)

Merged df shape: (15245000, 22)


## 3. Feature Engineering — Full Candidate Set

All features proposed by both teammates are built here. We intentionally include everything — including features that may be redundant — so that LightGBM can reveal the actual importance of each one.

In [ ]:
g  = df.groupby(['store_id', 'item_id'], observed=True)['sales']
pg = df.groupby(['store_id', 'item_id'], observed=True)['sell_price']

# --- Lag features ---
for lag in [1, 7, 14, 28, 35, 42, 364, 365]:
    df[f'lag_{lag}'] = g.shift(lag).astype('float32')

# --- Rolling mean features ---
for window in [3, 7, 28]:
    df[f'rmean_{window}'] = g.shift(1).rolling(window).mean().astype('float32')

# --- Rolling std ---
df['rolling_std_7'] = g.shift(1).rolling(7).std().astype('float32')

# --- Price features ---
df['price_lag_1']   = pg.shift(1).astype('float32')
df['price_change']  = (df['sell_price'] / df['price_lag_1']).astype('float32')
df['price_mean_7']  = pg.shift(1).rolling(7).mean().astype('float32')
df['price_mean_28'] = pg.shift(1).rolling(28).mean().astype('float32')
df['price_momentum'] = (df['price_mean_7'] / df['price_mean_28']).astype('float32')

item_mean_price     = df.groupby(['store_id', 'item_id'], observed=True)['sell_price'].transform('mean')
df['price_vs_mean'] = (df['sell_price'] / item_mean_price).astype('float32')

df.drop(columns=['price_lag_1', 'price_mean_28'], inplace=True)

# --- Calendar features ---
df['dayofweek']  = df['date'].dt.dayofweek.astype('int8')
df['month']      = df['date'].dt.month.astype('int8')
df['weekofyear'] = df['date'].dt.isocalendar().week.astype('int8')
df['is_weekend'] = (df['dayofweek'] >= 5).astype('int8')

high_impact = ['LaborDay', 'SuperBowl', 'Easter', 'Thanksgiving', 'Christmas']
df['is_high_impact_event'] = df['event_name_1'].isin(high_impact).astype('int8')
df['is_event']             = df['event_name_1'].notna().astype('int8')

# --- SNAP feature ---
df['snap'] = np.where(
    df['state_id'] == 'CA', df['snap_CA'],
    np.where(df['state_id'] == 'TX', df['snap_TX'], df['snap_WI'])
).astype('int8')

# --- Hierarchical features ---
df['store_sales_mean_7'] = (
    df.groupby(['store_id', 'date'], observed=True)['sales']
    .transform(lambda x: x.shift(1).rolling(7).mean())
).astype('float32')

df['cat_sales_mean_7'] = (
    df.groupby(['cat_id', 'date'], observed=True)['sales']
    .transform(lambda x: x.shift(1).rolling(7).mean())
).astype('float32')

print('Feature engineering done. df shape:', df.shape)

Feature engineering done. df shape: (15245000, 46)


## 4. Define Full Feature List

In [ ]:
FEATURES = [
    # Identity / categorical
    'item_id', 'dept_id', 'cat_id', 'state_id',
    # Lag features
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_35', 'lag_42', 'lag_364', 'lag_365',
    # Rolling mean
    'rmean_3', 'rmean_7', 'rmean_28',
    # Rolling std
    'rolling_std_7',
    # Price features
    'sell_price', 'price_change', 'price_mean_7', 'price_vs_mean', 'price_momentum',
    # Calendar
    'dayofweek', 'month', 'weekofyear', 'is_weekend',
    'is_event', 'is_high_impact_event',
    # SNAP
    'snap',
    # Hierarchical
    'store_sales_mean_7', 'cat_sales_mean_7',
]

TARGET = 'sales'
print(f'Total candidate features: {len(FEATURES)}')

## 5. Time-Based Train / Val Split

We use a **single store (CA_1)** as a representative to keep training fast. The last 28 days are held out as validation — this mirrors the actual forecasting task (predicting 28 days ahead).

**Why time-based and not random?**  
Lag and rolling features are computed from past sales. If we split randomly, a row from day 400 could end up in training, and a row from day 100 (which day 400's lag features depend on) could end up in validation. This leaks future information into training and makes feature importance scores unreliable.

In [ ]:
STORE    = 'CA_1'
store_df = df[df['store_id'] == STORE].copy()

val_date = store_df['date'].max() - pd.Timedelta(days=28)
train_df = store_df[store_df['date'] <= val_date].dropna(subset=FEATURES)
val_df   = store_df[store_df['date'] >  val_date].dropna(subset=FEATURES)

print(f'Train: {len(train_df):,} rows  ({train_df["date"].min().date()} ~ {train_df["date"].max().date()})')
print(f'Val  : {len(val_df):,} rows  ({val_df["date"].min().date()} ~ {val_df["date"].max().date()})')

## 6. Train LightGBM

In [ ]:
cat_cols = [c for c in FEATURES if train_df[c].dtype.name == 'category']

model = LGBMRegressor(
    objective='tweedie',
    tweedie_variance_power=1.1,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=127,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    train_df[FEATURES], train_df[TARGET],
    eval_set=[(val_df[FEATURES], val_df[TARGET])],
    eval_metric='rmse',
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print('\nBest iteration:', model.best_iteration_)

## 7. Feature Importance — Single Store (CA_1)

We report two types of importance:
- **Gain**: how much each feature reduces the loss when it is used in a split. This is the more meaningful metric — a high gain means the feature is actually informative.
- **Split**: how many times each feature is used to split. A feature with high split but low gain is being used a lot but not contributing much — a sign of noise or redundancy.

**Rule of thumb:** if a feature has gain < 0.5% AND split < 1%, it is a candidate for removal.

In [ ]:
def plot_importance(model, features, importance_type='gain', top_n=30, title_suffix=''):
    importance = model.booster_.feature_importance(importance_type=importance_type)
    fi_df = pd.DataFrame({'feature': features, 'importance': importance})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(10, 0.4 * len(fi_df) + 1))
    q75 = fi_df['importance'].quantile(0.75)
    colors = ['#e74c3c' if v > q75 else '#3498db' for v in fi_df['importance']]
    ax.barh(fi_df['feature'], fi_df['importance'], color=colors)
    ax.set_xlabel(f'Importance ({importance_type})', fontsize=12)
    ax.set_title(f'LightGBM Feature Importance — {importance_type.upper()} {title_suffix}', fontsize=13)
    ax.axvline(fi_df['importance'].mean(), color='gray', linestyle='--', alpha=0.6, label='mean')
    high_patch = mpatches.Patch(color='#e74c3c', label='Top 25%')
    low_patch  = mpatches.Patch(color='#3498db', label='Bottom 75%')
    ax.legend(handles=[high_patch, low_patch])
    plt.tight_layout()
    plt.show()
    return fi_df.sort_values('importance', ascending=False)

print('=== GAIN importance (primary metric) ===')
gain_df = plot_importance(model, FEATURES, 'gain', title_suffix=f'(store={STORE})')

In [ ]:
print('=== SPLIT importance ===')
split_df = plot_importance(model, FEATURES, 'split', title_suffix=f'(store={STORE})')

## 8. Summary Table

In [ ]:
summary = gain_df.rename(columns={'importance': 'gain'}).merge(
    split_df.rename(columns={'importance': 'split'}), on='feature'
)
summary['gain_rank']  = summary['gain'].rank(ascending=False).astype(int)
summary['split_rank'] = summary['split'].rank(ascending=False).astype(int)
summary['gain_pct']   = (summary['gain']  / summary['gain'].sum()  * 100).round(2)
summary['split_pct']  = (summary['split'] / summary['split'].sum() * 100).round(2)
summary['rank_diff']  = (summary['gain_rank'] - summary['split_rank']).abs()
summary = summary[['feature', 'gain_rank', 'gain_pct', 'split_rank', 'split_pct', 'rank_diff']]
summary = summary.sort_values('gain_rank')

print('Feature Importance Summary (sorted by Gain %):')
print(summary.to_string(index=False))

## 9. Multi-Store Validation

We repeat the analysis across CA_1, TX_1, and WI_1 (one per state) to check whether importance rankings are consistent. A feature that is low across all stores is a reliable candidate for removal.

In [ ]:
STORES_TO_CHECK  = ['CA_1', 'TX_1', 'WI_1']
all_importances  = {}

for store in STORES_TO_CHECK:
    sdf      = df[df['store_id'] == store].copy()
    val_date = sdf['date'].max() - pd.Timedelta(days=28)
    tr = sdf[sdf['date'] <= val_date].dropna(subset=FEATURES)
    vl = sdf[sdf['date'] >  val_date].dropna(subset=FEATURES)

    m = LGBMRegressor(
        objective='tweedie', tweedie_variance_power=1.1,
        n_estimators=300, learning_rate=0.05,
        num_leaves=127, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    cats = [c for c in FEATURES if tr[c].dtype.name == 'category']
    m.fit(tr[FEATURES], tr[TARGET],
          eval_set=[(vl[FEATURES], vl[TARGET])],
          categorical_feature=cats,
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])

    imp = m.booster_.feature_importance(importance_type='gain')
    all_importances[store] = imp / imp.sum() * 100
    print(f'Store {store} done (best iter: {m.best_iteration_})')

multi_store_df = pd.DataFrame(all_importances, index=FEATURES)
multi_store_df['mean_gain_pct'] = multi_store_df.mean(axis=1)
multi_store_df = multi_store_df.sort_values('mean_gain_pct', ascending=False)

print('\nMulti-Store Feature Importance (Gain %):')
print(multi_store_df.round(2).to_string())

In [ ]:
# Visualise multi-store comparison
fig, ax = plt.subplots(figsize=(12, 0.45 * len(FEATURES) + 1))
width   = 0.25
colors  = ['#3498db', '#e67e22', '#2ecc71']
sorted_features = multi_store_df.index.tolist()

for i, store in enumerate(STORES_TO_CHECK):
    vals = [multi_store_df.loc[f, store] for f in sorted_features]
    ax.barh([x - width + i * width for x in range(len(sorted_features))],
            vals, width, label=store, color=colors[i], alpha=0.8)

ax.set_yticks(range(len(sorted_features)))
ax.set_yticklabels(sorted_features)
ax.set_xlabel('Gain Importance (%)')
ax.set_title('Feature Importance Across Stores (Gain %)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Low-Importance Features (Drop Candidates)

In [ ]:
low_importance = summary[(summary['gain_pct'] < 0.5) & (summary['split_pct'] < 1.0)]

print('=== Low-importance features (candidates for removal) ===')
if len(low_importance) == 0:
    print('No clearly low-importance features found.')
else:
    print(low_importance[['feature', 'gain_pct', 'split_pct']].to_string(index=False))

print()
print('=== Top 10 most important features ===')
print(summary.head(10)[['feature', 'gain_pct', 'split_pct']].to_string(index=False))

## 11. Findings & Discussion

---

### 11.1 Overall Results

The top three features dominate overwhelmingly:

| Feature | Mean Gain % | What it captures |
|---|---|---|
| `rmean_28` | ~69% | 28-day average sales — recent demand level |
| `rmean_7` | ~16% | 7-day average sales — short-term trend |
| `item_id` | ~9% | Each item's baseline sales level |

Together these three account for **~93% of total gain**. This means the model is essentially learning: *"predict based on how much this item sold recently"*. Everything else is marginal by comparison.

---

### 11.2 Feature Redundancy Groups

The most important insight from this analysis is that many features are **redundant** — they encode overlapping information, so when one is present, the others contribute very little. Below are the main redundancy groups identified.

---

#### Group 1: Rolling Means absorb Lag Features

| Feature | Gain % |
|---|---|
| rmean_28 | 68.8% |
| rmean_7 | 15.8% |
| lag_1 | 0.54% |
| lag_7 | 0.09% |
| lag_28 | 0.08% |

`rmean_28` is the 28-day rolling average of past sales — it already contains the information from `lag_7`, `lag_14`, and `lag_28`. Once the model has access to the rolling mean, individual lag values have little additional signal. The only exception is `lag_1` (yesterday's sales), which captures an immediate signal that averaging smooths away.

**Important:** this does NOT mean lags are useless. It means *given that rolling means are already included*, lags are redundant. If rolling means were removed, lags would become important again.

---

#### Group 2: Rolling Means are partially redundant with each other

`rmean_3` (0.71%) contributes little when `rmean_7` and `rmean_28` are both present, since `rmean_7` already covers the 3-day window and more.

---

#### Group 3: Price Features are mutually redundant

| Feature | Gain % | What it measures |
|---|---|---|
| sell_price | 0.14% | Current price |
| price_momentum | 0.15% | 7-day vs 28-day price trend |
| price_change | 0.03% | Current vs previous price |
| price_mean_7 | 0.02% | Rolling average price |
| price_vs_mean | 0.03% | Current vs long-term mean price |

All five features are essentially answering the same question: *"has the price changed recently?"* The model only needs one representative — having all five means each one contributes minimally. Additionally, Walmart prices are relatively stable, so price signals are weak regardless.

---

#### Group 4: `is_weekend` is contained in `dayofweek`

`is_weekend` = 1 when `dayofweek` >= 5. Since `dayofweek` already encodes this information (and more), `is_weekend` adds no new signal. Its gain (0.19%) is entirely redundant with `dayofweek` (1.47%).

---

#### Group 5: `month` and `weekofyear` overlap

Both features describe "where we are in the year". `weekofyear` is a finer version of `month`, so they partially overlap. Neither performs well (0.11% and 0.17% respectively) — see Section 11.3 for why.

---

#### Group 6: Categorical ID hierarchy

`item_id` (9.1%) uniquely identifies each product. `dept_id` (0.13%) and `cat_id` (0.11%) are higher-level groupings whose information is entirely contained within `item_id`. Once the model knows the item, it doesn't gain additional signal from knowing its department or category.

---

### 11.3 Effect of the 500-Day Data Window

Using only the last 500 days affects different features differently:

| Feature | Why 500 days hurts it |
|---|---|
| `lag_364`, `lag_365` | Only ~135 valid rows (500 - 365). Almost entirely NaN. |
| `month` | Each month appears only 1-2 times — not enough to learn seasonal patterns. |
| `weekofyear` | 52 possible values, each appears ~1.4 times on average. |

**However**, this explanation does NOT apply to shorter lags like `lag_7` or `lag_28`. With 500 days of data, `lag_7` has 493 valid rows — plenty to learn from. Its low importance is due to redundancy with rolling means (Group 1 above), not data scarcity.

**Summary of root causes:**

| Root cause | Affected features |
|---|---|
| Redundancy with rolling means | `lag_7`, `lag_28`, `lag_35`, `lag_42` |
| Redundancy within price group | `price_change`, `price_mean_7`, `price_vs_mean` |
| Redundancy within calendar group | `is_weekend`, `month` (partial) |
| Insufficient data window (500 days) | `lag_364`, `lag_365`, `month`, `weekofyear` |
| Already captured by item_id | `dept_id`, `cat_id`, `state_id` |

---

### 11.4 Recommended Feature Set

Based on the importance analysis and redundancy findings:

**Keep:**
`rmean_28`, `rmean_7`, `rmean_3`, `item_id`, `dayofweek`, `rolling_std_7`, `lag_1`, `is_high_impact_event`, `store_sales_mean_7`, `lag_42`, `lag_35`, `lag_7`, `lag_28`, `price_momentum`, `sell_price`, `weekofyear`, `dept_id`, `month`

**Remove (low gain, redundant, or zero importance):**
`state_id` (0%), `cat_sales_mean_7` (0.02%), `price_mean_7` (0.02%), `price_change` (0.03%), `snap` (0.03%), `price_vs_mean` (0.03%), `is_event` (0.07%), `lag_14` (0.07%), `is_weekend` (redundant with dayofweek), `cat_id` (redundant with item_id)

**Consider with more data (currently hurt by 500-day window):**
`lag_364`, `lag_365` — worth testing if training window is extended to 1000+ days